# UNSW-NB15 — centralized XGBoost baseline trên Kaggle

Chạy cùng pipeline `unsw/` như trên máy local (cấu hình đã chốt trong CLAUDE.md):
7 class (`merged`), bỏ TTL, + feature tỉ lệ, class weight α=0.5.

**Chuẩn bị (một lần):**
1. **Dữ liệu**: *Add Input* → tìm `UNSW-NB15` → chọn dataset có `UNSW-NB15_1.csv … UNSW-NB15_4.csv` và `NUSW-NB15_features.csv`.
2. **Code**, chọn một cách:
   - *Upload*: trên máy chạy `zip -r kaggle_code.zip unsw NUSW-NB15_features.csv -x '*/__pycache__/*'` → Kaggle *Datasets → New* → upload zip → *Add Input* vào notebook này.
   - *GitHub*: bật *Internet* (Settings), điền `GIT_URL` ở cell cấu hình (repo private: thêm token vào *Add-ons → Secrets* tên `GITHUB_TOKEN`). Code phải đã được push.
3. **Accelerator**: GPU T4 hoặc P100 (XGBoost chạy `device='cuda'`). Không có GPU thì tự chạy CPU.

**Lưu ý so sánh**: kết quả GPU có thể lệch rất nhỏ so với CPU → mọi run trong một lượt so sánh phải cùng thiết bị.

In [ ]:
# ===== Cấu hình =====
import os, sys, glob, json, shutil, subprocess, time
from pathlib import Path

GIT_URL   = ''          # vd 'https://github.com/<owner>/cyberattack_detection.git' (để trống nếu dùng code upload)
GIT_REF   = 'main'
SPLIT_SEED = 42         # seed chia train/val/test (đổi để chạy B2: 42/43/44)
SMOKE     = os.environ.get('SMOKE') == '1'   # chạy thử nhanh (cắt train mỗi class, ít cây)
# b1 = tinh chỉnh (base + random search) rồi B2 (seed 43, 44) | fixed = 1 cấu hình cố định
# e3_10class = 1 cấu hình, 10 class gốc | c2b_8class = 1 cấu hình, gộp 3 class (DoS riêng)
# b1_10class = tinh chỉnh 10 class (thêm α vào không gian tìm) | fixed_10class = cấu hình tốt nhất của b1_10class
# mlp_10class = torch_mlp (baseline cho fed) trên GPU: α sweep seed 42, chọn trên val, rồi seed 43/44
EXPERIMENT = os.environ.get('EXPERIMENT', 'b1')
CONFIG     = json.loads(os.environ.get('CONFIG_JSON', 'null'))   # dict cấu hình cho fixed / e3_10class
assert EXPERIMENT in ('b1', 'fixed', 'e3_10class', 'c2b_8class', 'b1_10class', 'fixed_10class', 'mlp_10class'), EXPERIMENT
TUNE = EXPERIMENT in ('b1', 'b1_10class')
IS_MLP = EXPERIMENT == 'mlp_10class'
assert TUNE or IS_MLP or CONFIG, 'fixed / e3_10class / fixed_10class cần CONFIG'
SCHEME = {'e3_10class': 'original', 'b1_10class': 'original', 'fixed_10class': 'original', 'mlp_10class': 'original',
          'c2b_8class': 'merged3'}.get(EXPERIMENT, 'merged')
print('EXPERIMENT =', EXPERIMENT, '| SCHEME =', SCHEME, '| SPLIT_SEED =', SPLIT_SEED, '| CONFIG =', CONFIG)

INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input'))
WORK  = Path(os.environ.get('KAGGLE_WORK', '/kaggle/working'))
WORK.mkdir(parents=True, exist_ok=True)

# Cấu hình dữ liệu đã chốt (CLAUDE.md, "Quyết định đã chốt")
PREPARE_ARGS = ['--label-scheme', SCHEME,
                '--drop-features', 'sttl,dttl,ct_state_ttl',
                '--derived-features', 'all']
# XGBoost cơ sở (E8) — B1 sẽ thay đổi quanh cấu hình này
BASE_XGB = dict(n_estimators=500, max_depth=10, tree_lr=0.1, class_weight_alpha=0.5, xgb_params={})

In [ ]:
# ===== Tìm dữ liệu raw =====
hits = sorted(glob.glob(str(INPUT / '**' / 'UNSW-NB15_1.csv'), recursive=True))
assert hits, 'Không thấy UNSW-NB15_1.csv trong input — hãy Add Input dataset raw UNSW-NB15'
RAW_DIR = Path(hits[0]).parent
print('RAW_DIR =', RAW_DIR)
print(sorted(p.name for p in RAW_DIR.iterdir()))
assert len(list(RAW_DIR.glob('UNSW-NB15_[0-9].csv'))) == 4, 'Cần đủ 4 file UNSW-NB15_1..4.csv'
FEATURES_CSV = next(iter(glob.glob(str(INPUT / '**' / 'NUSW-NB15_features.csv'), recursive=True)), None)
print('features csv =', FEATURES_CSV)

In [ ]:
# ===== Lấy code =====
CODE = WORK / 'code'
if CODE.exists(): shutil.rmtree(CODE)
if GIT_URL:
    url = GIT_URL
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
        url = GIT_URL.replace('https://', f'https://{tok}@')
    except Exception:
        pass
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_REF, url, str(CODE)], check=True)
else:
    src = sorted(glob.glob(str(INPUT / '**' / 'unsw' / 'raw.py'), recursive=True))
    if not src:   # dataset có thể chứa file zip chưa giải nén
        zips = glob.glob(str(INPUT / '**' / '*.zip'), recursive=True)
        assert zips, 'Không thấy thư mục unsw/ (hoặc zip) trong input'
        shutil.unpack_archive(zips[0], WORK / 'code_unzip')
        src = sorted(glob.glob(str(WORK / 'code_unzip' / '**' / 'unsw' / 'raw.py'), recursive=True))
    shutil.copytree(Path(src[0]).parent.parent, CODE, ignore=shutil.ignore_patterns('__pycache__'))
# raw.py tìm NUSW-NB15_features.csv cạnh dữ liệu raw hoặc trong thư mục làm việc
if not (CODE / 'NUSW-NB15_features.csv').exists() and FEATURES_CSV:
    shutil.copy(FEATURES_CSV, CODE / 'NUSW-NB15_features.csv')
sys.path.insert(0, str(CODE))
print(sorted(p.name for p in (CODE / 'unsw').glob('*.py')))

In [ ]:
# ===== Môi trường =====
import numpy as np, pandas as pd, sklearn, xgboost
print('python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__,
      '| sklearn', sklearn.__version__, '| xgboost', xgboost.__version__)
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True, timeout=20).stdout.strip()
except Exception:
    gpu = ''
DEVICE = 'cuda' if gpu else 'cpu'
print('GPU:', gpu or 'không có', '→ XGBoost device =', DEVICE)
print('CPU cores:', os.cpu_count())
print(subprocess.run('free -g 2>/dev/null | head -2', shell=True, capture_output=True, text=True).stdout)

def sh(args, log=None):
    # chạy `python -m unsw...` trong thư mục code; in phần cuối log nếu lỗi
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', *args], cwd=CODE, capture_output=True, text=True)
    if log: Path(log).write_text(r.stdout + r.stderr)
    if r.returncode:
        print((r.stdout + r.stderr)[-3000:]); raise RuntimeError(f'failed: {args[:2]}')
    return time.time() - t0

## 1. Prepare dữ liệu (chia theo nhóm feature vector, 70/10/20)
Khoảng 1–2 phút, RAM đỉnh ~9 GB. Cùng `SPLIT_SEED` → cùng `test_ids` với máy local.

In [ ]:
EXP  = WORK / 'experiments'; EXP.mkdir(parents=True, exist_ok=True)
# data.npz (~150 MB) goes outside /kaggle/working so it is not saved/downloaded as notebook output
DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/tmp/unsw_data' if str(WORK).startswith('/kaggle') else str(EXP)))
DATA = DATA_ROOT / f'data_{SCHEME}_s{SPLIT_SEED}'
if not (DATA / 'data.npz').exists():
    extra = ['--limit-per-class', '20000'] if SMOKE else []
    t = sh(['unsw.run', 'prepare', '--raw-dir', str(RAW_DIR), '--seed', str(SPLIT_SEED),
            '--output', str(DATA), *PREPARE_ARGS, *extra], log=EXP / f'prepare_s{SPLIT_SEED}.log')
    print(f'prepare {t:.0f}s')
audit = json.loads((DATA / 'audit.json').read_text())
shutil.copy(DATA / 'audit.json', EXP / f'audit_{SCHEME}_s{SPLIT_SEED}.json')
print({k: audit[k] for k in ['label_scheme', 'dropped_features', 'fit_rows', 'validation_rows', 'test_rows', 'input_dim']})
print('test counts:', audit['test_counts'])

In [ ]:
# ===== Nạp dữ liệu vào kernel (để chấm theo từng class) =====
from unsw.data import load
D = load(DATA / 'data.npz'); NAMES = list(D['labels']); K = len(NAMES)
import hashlib
np.savez_compressed(EXP / f'labels_{SCHEME}_s{SPLIT_SEED}.npz', y_val=D['y_val'], y_test=D['y_test'],
                    y_orig_val=D['y_orig_val'], y_orig_test=D['y_orig_test'], names=np.asarray(NAMES), orig_names=D['orig_names'],
                    test_ids_sha=hashlib.sha256('\n'.join(D['test_ids']).encode()).hexdigest())
from sklearn.metrics import precision_recall_fscore_support as prfs, confusion_matrix

def per_class(y, p):
    P, R, F, S = prfs(y, p, labels=range(K), zero_division=0)
    return pd.DataFrame({'precision': P, 'recall': R, 'f1': F, 'n': S}, index=NAMES)

def run_xgb(name, n_estimators, max_depth, tree_lr, class_weight_alpha, xgb_params, data_dir=None, dd=None, seed=None):
    # train 1 cấu hình qua CLI (giống hệt local), rồi chấm val/test theo từng class
    data_dir = data_dir or DATA; dd = D if dd is None else dd; seed = SPLIT_SEED if seed is None else seed
    out = EXP / 'runs' / name
    if not (out / 'results.json').exists():
        args = ['unsw.run', 'baseline', '--baseline-model', 'xgboost', '--data', str(data_dir / 'data.npz'),
                '--output', str(out), '--seed', str(seed), '--n-jobs', '-1', '--xgb-device', DEVICE,
                '--n-estimators', str(30 if SMOKE else n_estimators), '--max-depth', str(max_depth),
                '--tree-lr', str(tree_lr), '--class-weight-alpha', str(class_weight_alpha)]
        if xgb_params: args += ['--xgb-params', json.dumps(xgb_params)]
        t = sh(args, log=str(out) + '.log'); print(f'{name}: trained in {t:.0f}s')
    if not (out / 'predictions.npz').exists():
        import joblib
        est = joblib.load(out / 'baseline.joblib')['model']
        np.savez(out / 'predictions.npz', val=est.predict_proba(dd['X_val']).argmax(1),
                 test=est.predict_proba(dd['X_test']).argmax(1))
    pr = np.load(out / 'predictions.npz')
    return per_class(dd['y_val'], pr['val']), per_class(dd['y_test'], pr['test'])

## 2b. torch_mlp trên GPU (chỉ khi EXPERIMENT = mlp_10class)
Baseline centralized cho fed: cùng `model()`, Adam, CE + class weight toàn cục, cùng `data.npz`, chạy trên GPU như fed (cloud).
Hyperparameter như B3 local: 512-256-128-64, Adam lr 1e-3, batch 256, 30 epoch (chọn epoch theo val macro-F1).
α ∈ {0, 0.25, 0.5} ở split seed 42 → **chọn α trên val** (F1 trung bình 8 class khó, như b1_10class) → chạy α đó ở seed 43/44.
`CONFIG` (tuỳ chọn) ghi đè: `{"alphas": [...], "seeds": [...], "epochs": N}`.

In [ ]:
if IS_MLP:
    MCFG = {'alphas': [0, 0.25, 0.5], 'seeds': [43, 44], 'epochs': 30, **(CONFIG or {})}
    if SMOKE: MCFG['epochs'] = 2
    MLP_ARGS = ['--epochs', str(MCFG['epochs']), '--batch-size', '256', '--hidden', '512', '--hidden-layers', '4', '--lr', '0.001']
    # predictions in a separate process (never load torch next to xgboost in this kernel process)
    PRED = WORK / 'predict_mlp.py'
    PRED.write_text("""import os, sys, numpy as np, torch
sys.path.insert(0, os.getcwd())
from unsw.data import load
from unsw.model import model, predict
run, data = sys.argv[1], sys.argv[2]; d = load(data)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load(run + '/best.pt', weights_only=False, map_location='cpu'); c = ck['config']
net = model(ck['input_dim'], c['hidden'], c['hidden_layers'], ck['n_classes']); net.load_state_dict(ck['state_dict']); net.to(dev)
np.savez(run + '/predictions.npz', val=predict(net, d['X_val'], dev).argmax(1), test=predict(net, d['X_test'], dev).argmax(1))
""")
    def run_mlp(name, alpha, data_dir, dd, seed):
        out = EXP / 'runs' / name
        if not (out / 'results.json').exists():
            t = sh(['unsw.run', 'centralized', '--data', str(data_dir / 'data.npz'), '--output', str(out), '--seed', str(seed),
                    '--device', DEVICE, '--class-weight-alpha', str(alpha), *MLP_ARGS], log=str(out) + '.log')
            print(f'{name}: trained in {t:.0f}s')
        if not (out / 'predictions.npz').exists():
            r = subprocess.run([sys.executable, str(PRED), str(out), str(data_dir / 'data.npz')], cwd=CODE, capture_output=True, text=True)
            if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError('predict failed: ' + name)
        pr = np.load(out / 'predictions.npz')
        return pr, per_class(dd['y_val'], pr['val']), per_class(dd['y_test'], pr['test'])

    HARD = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Reconnaissance', 'Shellcode', 'Worms']
    iN = NAMES.index('Normal'); mlp_rows = []; sel = []
    def collect(seed, alpha, pr, v, t_, dd):
        for split, tab in [('val', v), ('test', t_)]:
            y, q = dd['y_' + split], pr[split]
            for c in NAMES:
                mlp_rows.append({'seed': seed, 'alpha': alpha, 'split': split, 'cls': c, **tab.loc[c].to_dict(),
                                 'tp': int(round(tab.recall[c] * tab.n[c]))})
            mlp_rows.append({'seed': seed, 'alpha': alpha, 'split': split, 'cls': '_false_alarm', 'tp': int(((y == iN) & (q != iN)).sum())})
            mlp_rows.append({'seed': seed, 'alpha': alpha, 'split': split, 'cls': '_missed', 'tp': int(((y != iN) & (q == iN)).sum())})
    for a in MCFG['alphas']:
        pr, v, t_ = run_mlp(f'mlp_a{a}_s{SPLIT_SEED}', a, DATA, D, SPLIT_SEED)
        collect(SPLIT_SEED, a, pr, v, t_, D)
        sel.append({'alpha': a, **{f'val_f1_{c}': v.f1[c] for c in NAMES}, 'hard_mean_f1': v.f1[HARD].mean(),
                    'val_false_alarm': int(((D['y_val'] == iN) & (pr['val'] != iN)).sum()),
                    'val_missed': int(((D['y_val'] != iN) & (pr['val'] == iN)).sum()),
                    'train_s': json.loads((EXP / 'runs' / f'mlp_a{a}_s{SPLIT_SEED}' / 'results.json').read_text())['train_s']})
    SEL = pd.DataFrame(sel).sort_values('hard_mean_f1', ascending=False); SEL.to_csv(EXP / 'mlp_alpha_selection.csv', index=False)
    display(SEL.round(3))
    A_BEST = float(SEL.iloc[0].alpha); print('Chọn α =', A_BEST, '(trên val, seed', SPLIT_SEED, ')')
    (EXP / 'mlp_best.json').write_text(json.dumps({'alpha': A_BEST, 'mlp_args': MLP_ARGS, 'device': DEVICE}, indent=2))
    for s2 in MCFG['seeds']:
        data2 = DATA_ROOT / f'data_{SCHEME}_s{s2}'
        if not (data2 / 'data.npz').exists():
            extra = ['--limit-per-class', '20000'] if SMOKE else []
            t = sh(['unsw.run', 'prepare', '--raw-dir', str(RAW_DIR), '--seed', str(s2), '--output', str(data2),
                    *PREPARE_ARGS, *extra], log=EXP / f'prepare_s{s2}.log')
            print(f'prepare s{s2}: {t:.0f}s')
        D2 = load(data2 / 'data.npz')
        np.savez_compressed(EXP / f'labels_{SCHEME}_s{s2}.npz', y_val=D2['y_val'], y_test=D2['y_test'],
                            y_orig_val=D2['y_orig_val'], y_orig_test=D2['y_orig_test'], names=np.asarray(NAMES), orig_names=D2['orig_names'],
                            test_ids_sha=hashlib.sha256('\n'.join(D2['test_ids']).encode()).hexdigest())
        pr, v, t_ = run_mlp(f'mlp_a{A_BEST}_s{s2}', A_BEST, data2, D2, s2)
        collect(s2, A_BEST, pr, v, t_, D2); del D2
    MLPT = pd.DataFrame(mlp_rows); MLPT.to_csv(EXP / 'mlp_per_class.csv', index=False)
    m = MLPT[(MLPT.alpha == A_BEST) & ~MLPT.cls.str.startswith('_')].groupby(['split', 'cls']).f1.agg(['mean', 'std']).round(3)
    display(m.unstack('split').loc[NAMES])
else:
    print('bỏ qua torch_mlp (EXPERIMENT =', EXPERIMENT + ')')

## 2. Cấu hình cơ sở (b1) hoặc cấu hình cố định (fixed / e3_10class)
b1: chạy lại E8 làm mốc. fixed / e3_10class: chạy đúng `CONFIG` (thường là cấu hình tốt nhất của B1).

In [ ]:
if IS_MLP:
    print('bỏ qua XGBoost (EXPERIMENT = mlp_10class)')
elif TUNE:
    base_val, base_test = run_xgb('base', **BASE_XGB)
    BEST_RUN = 'base'; best_cfg = BASE_XGB
else:
    cfg = {**BASE_XGB, **CONFIG}
    base_val, base_test = run_xgb('fixed', **cfg)
    BEST_RUN = 'fixed'
    (EXP / 'fixed_config.json').write_text(json.dumps(cfg, indent=2))
if not IS_MLP:
    print('VAL'); display(base_val.round(3))
    print('TEST'); display(base_test.round(3))

## 3. B1 — tinh chỉnh XGBoost trên **validation**
Random search quanh cấu hình cơ sở (E8). Mỗi cấu hình là một run riêng (chạy lại cell sẽ bỏ qua run đã xong).
20 cấu hình cây giống hệt B1 7 class (cùng `rng(0)`); `b1_10class` bốc thêm α ∈ {0.25, 0.5, 0.75} bằng `rng(1)` riêng.

**Luật chọn (chỉ trên val):**
- `b1`: loại cấu hình làm bất kỳ class nào giảm F1 > 0.01 so với `base`; xếp theo F1 trung bình của Fuzzers, Recon, Shellcode, Worms.
- `b1_10class`: ràng buộc 0.01 chỉ áp cho class có ≥ 1.000 mẫu val (class nhỏ: 1 mẫu đã lệch nhiều → xem số mẫu đúng `val_tp_*`);
  xếp theo F1 trung bình của 8 class khó: Analysis, Backdoor, DoS, Exploits, Fuzzers, Recon, Shellcode, Worms.
- Bảng kèm số Normal bị báo nhầm (`val_false_alarm`) và attack bị bỏ sót (`val_missed`) để thấy cấu hình trượt trên đường đánh đổi.

Test chỉ để báo cáo cấu hình được chọn. `base` có trong bảng (nếu không cấu hình nào hơn thì giữ `base`).

In [ ]:
if not TUNE:
    print('bỏ qua B1 (EXPERIMENT =', EXPERIMENT + ')')
else:
    N_TRIALS = 3 if SMOKE else 20
    rng = np.random.default_rng(0)
    rng_a = np.random.default_rng(1)   # α bốc riêng → 20 cấu hình cây giữ nguyên như B1 7 class
    SPACE = dict(max_depth=[6, 8, 10, 12], tree_lr=[0.05, 0.1, 0.2], n_estimators=[300, 500, 800],
                 min_child_weight=[1, 3, 5], subsample=[0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0],
                 reg_lambda=[0.5, 1.0, 5.0])
    ALPHAS = [0.25, 0.5, 0.75] if EXPERIMENT == 'b1_10class' else [BASE_XGB['class_weight_alpha']]
    pick = lambda k: SPACE[k][rng.integers(len(SPACE[k]))]
    trials = []
    for i in range(N_TRIALS):
        cfg = dict(n_estimators=int(pick('n_estimators')), max_depth=int(pick('max_depth')), tree_lr=float(pick('tree_lr')),
                   class_weight_alpha=float(ALPHAS[rng_a.integers(len(ALPHAS))]),
                   xgb_params=dict(min_child_weight=int(pick('min_child_weight')), subsample=float(pick('subsample')),
                                   colsample_bytree=float(pick('colsample_bytree')), reg_lambda=float(pick('reg_lambda'))))
        trials.append((f'b1_{i:02d}', cfg))
    (EXP / 'b1_trials.json').write_text(json.dumps(trials, indent=1))

    if EXPERIMENT == 'b1_10class':
        HARD = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Reconnaissance', 'Shellcode', 'Worms']
        BIG = [c for c in NAMES if base_val.n[c] >= 1000]
    else:
        HARD = ['Fuzzers', 'Reconnaissance', 'Shellcode', 'Worms']; BIG = NAMES
    SMALL = [c for c in NAMES if c not in BIG]
    print('ràng buộc 0.01 cho:', BIG, '| class nhỏ (xem số mẫu đúng):', SMALL)
    iN = NAMES.index('Normal')
    def row(name, cfg, val):
        pv = np.load(EXP / 'runs' / name / 'predictions.npz')['val']; yv = D['y_val']
        tp = (val.recall * val.n).round().astype(int)
        return {'run': name, **{k: x for k, x in cfg.items() if k != 'xgb_params'}, **cfg['xgb_params'],
                **{f'val_f1_{c}': val.f1[c] for c in NAMES}, **{f'val_tp_{c}': tp[c] for c in SMALL},
                'val_false_alarm': int(((yv == iN) & (pv != iN)).sum()), 'val_missed': int(((yv != iN) & (pv == iN)).sum()),
                'max_drop_vs_base': (base_val.f1[BIG] - val.f1[BIG]).max(), 'hard_mean_f1': val.f1[HARD].mean()}
    rows = [row('base', BASE_XGB, base_val)]
    for name, cfg in trials:
        val, _ = run_xgb(name, **cfg)
        rows.append(row(name, cfg, val))
    B1 = pd.DataFrame(rows)
    B1['eligible'] = B1.max_drop_vs_base <= 0.01
    B1 = B1.sort_values(['eligible', 'hard_mean_f1'], ascending=False)
    B1.to_csv(EXP / 'b1_results.csv', index=False)
    display(B1.round(3).head(10))

In [ ]:
if TUNE:
    # ===== Cấu hình được chọn: F1 từng class (val để chọn, test để báo cáo) =====
    best = B1.iloc[0]
    print('Chọn:', best.run, '| hard_mean_f1 val', round(best.hard_mean_f1, 3), 'vs base', round(base_val.f1[HARD].mean(), 3))
    best_cfg = BASE_XGB if best.run == 'base' else dict(trials)[best.run]
    bv, bt = run_xgb(best.run, **best_cfg)
    cmp = pd.DataFrame({'val_base': base_val.f1, 'val_best': bv.f1, 'test_base': base_test.f1, 'test_best': bt.f1})
    display(cmp.round(3))
    (EXP / 'b1_best.json').write_text(json.dumps({'run': best.run, 'config': best_cfg}, indent=2))
    BEST_RUN = best.run

In [ ]:
if not IS_MLP:
    # ===== Confusion matrix (test) của cấu hình được chọn =====
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    pr = np.load(EXP / 'runs' / BEST_RUN / 'predictions.npz')['test']
    cm = confusion_matrix(D['y_test'], pr, labels=range(K)); pct = cm / cm.sum(1, keepdims=True) * 100
    cmap = LinearSegmentedColormap.from_list('b', ['#ffffff', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])
    fig, ax = plt.subplots(figsize=(9.6, 6.8), dpi=150)
    im = ax.imshow(pct, cmap=cmap, vmin=0, vmax=100, aspect='auto')
    for i in range(K):
        for j in range(K):
            ax.text(j, i, '0' if cm[i, j] == 0 else f'{pct[i, j]:.1f}%\n{cm[i, j]:,}', ha='center', va='center', fontsize=7,
                    color='white' if pct[i, j] > 55 else ('#1f1f1d' if cm[i, j] else '#b5b4af'), fontweight='bold' if i == j else 'normal')
    ax.set_xticks(range(K), NAMES, rotation=20); ax.set_yticks(range(K), [f'{n} (n={s:,})' for n, s in zip(NAMES, cm.sum(1))])
    ax.xaxis.tick_top(); ax.set_xlabel('Dự đoán'); ax.xaxis.set_label_position('top'); ax.set_ylabel('Thật')
    for s in ax.spines.values(): s.set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.03).set_label('% của hàng (recall)')
    ax.set_title(f'XGBoost {EXPERIMENT}/{BEST_RUN} — test, {SCHEME}, split seed {SPLIT_SEED}, {DEVICE}', loc='left', pad=40, fontweight='bold')
    fig.tight_layout(); fig.savefig(EXP / f'confusion_{EXPERIMENT}_{BEST_RUN}.png', facecolor='white'); plt.show()

## 3b. B2 — cấu hình tốt nhất và base trên split seed 43, 44 (chỉ khi EXPERIMENT = b1)
Đo độ ổn định khi đổi cách chia (đổi seed model vô nghĩa vì XGBoost tất định). Không chọn gì ở đây.


In [ ]:
B2_SEEDS = [43, 44] if EXPERIMENT == 'b1' else []
b2 = []
for s2 in B2_SEEDS:
    data2 = DATA_ROOT / f'data_{SCHEME}_s{s2}'
    if not (data2 / 'data.npz').exists():
        extra = ['--limit-per-class', '20000'] if SMOKE else []
        t = sh(['unsw.run', 'prepare', '--raw-dir', str(RAW_DIR), '--seed', str(s2), '--output', str(data2),
                *PREPARE_ARGS, *extra], log=EXP / f'prepare_s{s2}.log')
        print(f'prepare s{s2}: {t:.0f}s')
    D2 = load(data2 / 'data.npz')
    np.savez_compressed(EXP / f'labels_{SCHEME}_s{s2}.npz', y_val=D2['y_val'], y_test=D2['y_test'],
                        y_orig_val=D2['y_orig_val'], y_orig_test=D2['y_orig_test'], names=np.asarray(NAMES), orig_names=D2['orig_names'],
                        test_ids_sha=hashlib.sha256('\n'.join(D2['test_ids']).encode()).hexdigest())
    for tag, cfg in [('base', BASE_XGB), (BEST_RUN, best_cfg)]:
        v, t_ = run_xgb(f'{tag}_s{s2}', **cfg, data_dir=data2, dd=D2, seed=s2)
        for split, tab in [('val', v), ('test', t_)]:
            for c in NAMES:
                b2.append({'seed': s2, 'run': tag, 'split': split, 'cls': c, **tab.loc[c].to_dict()})
    del D2
# seed 42 rows from the main run, for a complete 3-split table
if B2_SEEDS:
    for tag in ['base', BEST_RUN]:
        pr = np.load(EXP / 'runs' / tag / 'predictions.npz')
        for split in ['val', 'test']:
            tab = per_class(D['y_' + split], pr[split])
            for c in NAMES:
                b2.append({'seed': SPLIT_SEED, 'run': tag, 'split': split, 'cls': c, **tab.loc[c].to_dict()})
    B2 = pd.DataFrame(b2); B2.to_csv(EXP / 'b2_per_class.csv', index=False)
    summ = B2.groupby(['run', 'split', 'cls']).f1.agg(['mean', 'std']).round(3)
    display(summ.unstack(['run', 'split']).loc[NAMES])
else:
    print('bỏ qua B2 (EXPERIMENT =', EXPERIMENT + ')')

## 4. Tải kết quả về
Gói mọi thứ trừ dữ liệu/model lớn (`.npz` dữ liệu, `.joblib`) → *Output* của notebook.

In [ ]:
keep = [p for p in EXP.rglob('*') if p.is_file() and p.suffix not in ('.joblib',) and p.name != 'data.npz']
bundle = WORK / f'results_{EXPERIMENT}_s{SPLIT_SEED}.zip'
import zipfile
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep: z.write(p, p.relative_to(WORK))
print(bundle, f'{bundle.stat().st_size/1e6:.1f} MB,', len(keep), 'files')